<a href="https://colab.research.google.com/github/ac5589/Pytorch-tutorial-Youtube/blob/main/Convolutional_neural_network.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [27]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
%matplotlib inline


In [28]:
# Before we import the data we need to transform the data

# Convert MNIST image files into a tensor for 4 dimensions ( Num images, height, Width and color channel)

transform = transforms.ToTensor()

In [35]:
from logging import root

# Train the data
train_data = datasets.MNIST(root = '/cnn_data', train=True, download=True, transform=transform)

In [36]:
# Test the data
test_data = datasets.MNIST(root = '/cnn_data', train=False, download=True, transform=transform)

In [37]:
# Print the train data
train_data

Dataset MNIST
    Number of datapoints: 60000
    Root location: /cnn_data
    Split: Train
    StandardTransform
Transform: ToTensor()

In [38]:
# Print the test data
test_data

Dataset MNIST
    Number of datapoints: 10000
    Root location: /cnn_data
    Split: Test
    StandardTransform
Transform: ToTensor()

In [39]:
# We need to create a loader for the dataset
# First set the batch size for images

train_loader = DataLoader(train_data, batch_size=10, shuffle=True)
test_loader = DataLoader(test_data, batch_size=10, shuffle=False)

# Let us define the convolution NN model

# describe the CNN layers and what they are doing

# Set up a 2 layer convolutional layer

conv1 = nn.Conv2d(1, 6, 3, 1)
conv2 = nn.Conv2d(6, 16, 3, 1)

In [43]:
# Grab the first MNIST image

for i, (X_Train, y_train) in enumerate(train_data):
  print(f'{i}')
  break
X_Train.shape

0


torch.Size([1, 28, 28])

In [44]:
x = X_Train.view(1, 1, 28, 28)

In [46]:
# Perfom the first convolution
x = F.relu(conv1(x)) # Rectified linear unit for our activation

In [47]:
# What is the shape of x
# 1 Image
# 6 is the number of the filter
# 26 x 26 is the size of the feature MAP
x.shape

torch.Size([1, 6, 26, 26])

In [48]:
# Pass through the pooling layer
x = F.max_pool2d(x, 2, 2) # Kernel of 2 and stride of 2
x.shape

torch.Size([1, 6, 13, 13])

In [49]:
# Do the second convolution layer
# We lost two pixels since we did not set the padding
x = F.relu(conv2(x))
x.shape

torch.Size([1, 16, 11, 11])

In [50]:
# Pass though the pooling layer
x = F.max_pool2d(x, 2, 2)
x.shape

torch.Size([1, 16, 5, 5])

In [52]:
# Build the model for the CNN

# Model the class

class convolutionalNetwork(nn.Module):
  def __init__(self):
    super().__init__()
    self.conv1 = nn.Conv2d(1, 6, 3, 1)
    self.conv2 = nn.Conv2d(6, 16, 3, 1)

    # Fully connected layer
    self.fc1 = nn.Linear(5*5*16, 120)
    self.fc2 = nn.Linear(120, 84)
    self.fc3 = nn.Linear(84, 10)

  def forward(self, X):
    X = F.relu(self.conv1(X))
    X = F.max_pool2d(X, 2, 2)
    X = F.relu(self.conv2(X))
    X = F.max_pool2d(X, 2, 2)

    # Re-View to flatten the output
    X = X.view(-1, 16 * 5 * 5)

    # finally we need our fully conencted layer
    X = F.relu(self.fc1(X))
    X = F.relu(self.fc2(X))
    X = self.fc3(X)

    return F.log_softmax(X, dim=1)

In [53]:
# Now we need to create an instance for our model

# Create the seed. We can use any number
torch.manual_seed(41)

model = convolutionalNetwork()

In [54]:
model

convolutionalNetwork(
  (conv1): Conv2d(1, 6, kernel_size=(3, 3), stride=(1, 1))
  (conv2): Conv2d(6, 16, kernel_size=(3, 3), stride=(1, 1))
  (fc1): Linear(in_features=400, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=10, bias=True)
)

In [55]:
# Loss function optimizer

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr = 0.001)